In [ ]:
# %%
# 00_feature_engineering.py
# ===========================================================================
# Builds the monthly link-level modelling panel from final_dataset.parquet.
#
# KEY DESIGN DECISIONS (documented for thesis §3)
# ─────────────────────────────────────────────────
#
# TARGET VARIABLE
#   This script saves proportion_disrupted (continuous) to the panel parquet.
#   The binary label Significant_Disruption is created in 01_tier1_model.py,
#   AFTER the temporal split, using the 50th percentile of
#   proportion_disrupted computed on TRAINING rows only.
#   This prevents any future information from leaking into the label.
#
#   proportion_disrupted_{e,t} = disrupted_days_{e,t} / rides_planned_{e,t}
#
# TOPOLOGY LEAKAGE FIX
#   G_static is built using rides_planned from TRAINING months only
#   (first TRAIN_FRAC of all months). The same constant must be used
#   in 01_tier1_model.py.
#
# LAG / ROLLING FEATURES
#   All use shift(1) so month t is never included in its own window.
# ===========================================================================

import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR   = Path("../../data/processed")
INPUT_FILE = DATA_DIR / "final_dataset.parquet"
OUT_FILE   = DATA_DIR / "modelling_panel_THRESHOLD.parquet"

# ---------------------------------------------------------------------------
# Constants — must match 01_tier1_model.py
# ---------------------------------------------------------------------------
TRAIN_FRAC = 0.70   # fraction of months used for training
MIN_RIDES  = 4      # minimum planned rides to keep a link-month (Kämpere, 2025)


# %%
# ---------------------------------------------------------------------------
# 1. Load & parse dates
# ---------------------------------------------------------------------------
print("Loading data ...")
df = pd.read_parquet(INPUT_FILE)
df["Date"]      = pd.to_datetime(df["Date"])
df["YearMonth"] = df["Date"].dt.to_period("M")


# %%
# ---------------------------------------------------------------------------
# 2. Clean weather columns
# ---------------------------------------------------------------------------
weather_cols = ["DR", "RH", "SQ", "TG", "TN", "TX", "RHX", "FHVEC", "VVX", "T10N"]
for col in weather_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.strip(), errors="coerce"
        )


# %%
# ---------------------------------------------------------------------------
# 3. Align cancellations with no disruption_count entry
#    (data-quality correction — unchanged from original)
# ---------------------------------------------------------------------------
if {"Final arrival cancelled", "Completely cancelled", "Disruption Count"}.issubset(df.columns):
    mask = (
        (df["Final arrival cancelled"].astype(int) > 0) &
        (df["Completely cancelled"].astype(int)    > 0) &
        (df["Disruption Count"]                    == 0)
    )
    df.loc[mask, "Disrupted"]        = True
    df.loc[mask, "Disruption Count"] = df.loc[mask, "Final arrival cancelled"]

df["Disrupted"] = df["Disrupted"].astype(int)


# %%
# ---------------------------------------------------------------------------
# 4. Identify SES columns
# ---------------------------------------------------------------------------
ses_cols = [
    c for c in df.columns
    if c.startswith(("source_", "target_")) and c not in ("source", "target")
]


# %%
# ---------------------------------------------------------------------------
# 5. Aggregate to monthly link-level panel
#
#    Disrupted     -> SUM (count of disrupted days; used to compute proportion)
#    rides_planned -> SUM
#    weather       -> mean
#    SES           -> mean
# ---------------------------------------------------------------------------
agg_ops = {
    "Disrupted":     "sum",
    "Rides planned": "sum",
}
for col in weather_cols:
    if col in df.columns:
        agg_ops[col] = "mean"
for col in ses_cols:
    if col in df.columns:
        agg_ops[col] = "mean"

panel = (
    df.groupby(["source", "target", "YearMonth"])
      .agg(agg_ops)
      .reset_index()
      .rename(columns={
          "Rides planned": "rides_planned",
          "Disrupted":     "disrupted_days",   # raw monthly disruption count
      })
)


# %%
# ---------------------------------------------------------------------------
# 6. Minimum-rides filter (Kämpere & Alsahag, 2025)
#    Drop link-months with fewer than MIN_RIDES planned rides.
# ---------------------------------------------------------------------------
before = len(panel)
panel  = panel[panel["rides_planned"] >= MIN_RIDES].copy()
after  = len(panel)
print(
    f"Min-rides filter (>= {MIN_RIDES}): {before} -> {after} rows "
    f"({before - after} dropped, {100 * (before - after) / before:.1f} %)"
)


# %%
# ---------------------------------------------------------------------------
# 7. Disruption proportion (continuous)
#    Saved to parquet as-is. The binary threshold is applied in
#    01_tier1_model.py after the temporal split.
# ---------------------------------------------------------------------------
panel["proportion_disrupted"] = (
    panel["disrupted_days"] / panel["rides_planned"]
).clip(0, 1)   # guard against data-quality edge cases > 1


# %%
# ---------------------------------------------------------------------------
# 8. Active service days per month
# ---------------------------------------------------------------------------
stop_counts = (
    df.groupby(["source", "target", "YearMonth"])["Date"]
      .nunique()
      .reset_index()
      .rename(columns={"Date": "stop_count"})
)
panel = panel.merge(stop_counts, on=["source", "target", "YearMonth"], how="left")


# %%
# ---------------------------------------------------------------------------
# 9. Derived features
# ---------------------------------------------------------------------------

# Remoteness Index
for prefix in ("source_", "target_"):
    hw  = f"{prefix}Dist_Highway_Entrance"
    mts = f"{prefix}Dist_Major_Transfer_Station"
    if hw in panel.columns and mts in panel.columns:
        panel[f"{prefix}Remoteness_Index"] = (panel[hw] + panel[mts]) / 2


# %%
# ---------------------------------------------------------------------------
# 10. Lag & rolling features
#     All use shift(1): month t is NEVER included in its own window.
#     Based on disrupted_days (raw count, no threshold applied).
# ---------------------------------------------------------------------------
panel = panel.sort_values(["source", "target", "YearMonth"]).reset_index(drop=True)

# Binary lag: was there at least one disrupted day in the previous month(s)?
for lag in [1, 2, 3]:
    panel[f"disrupted_lag{lag}"] = (
        panel.groupby(["source", "target"])["disrupted_days"]
             .transform(lambda x, l=lag: (x.shift(l) > 0).astype(float))
    )

# Rolling disruption frequency — entirely in the past
panel["delay_freq_3m"] = (
    panel.groupby(["source", "target"])["disrupted_days"]
         .transform(
             lambda x: (x.shift(1) > 0).astype(float)
                        .rolling(3, min_periods=1).mean()
         )
)
panel["delay_freq_6m"] = (
    panel.groupby(["source", "target"])["disrupted_days"]
         .transform(
             lambda x: (x.shift(1) > 0).astype(float)
                        .rolling(6, min_periods=1).mean()
         )
)


# %%
# ---------------------------------------------------------------------------
# 11. Topology features — built on TRAINING months only
#
#     Building G_static on all months would leak test-period operational
#     volumes into the centrality features. We replicate the 70 % training
#     cut here so the graph only sees training-period traffic.
# ---------------------------------------------------------------------------
periods_all  = sorted(panel["YearMonth"].unique())
n_train      = int(len(periods_all) * TRAIN_FRAC)
train_months = set(periods_all[:n_train])

print(
    f"Topology graph: built on {n_train}/{len(periods_all)} months "
    f"(up to and including {periods_all[n_train - 1]})"
)

edge_weights = (
    panel[panel["YearMonth"].isin(train_months)]
      .groupby(["source", "target"])["rides_planned"]
      .sum()
      .reset_index()
      .rename(columns={"rides_planned": "total_planned"})
)

G_static = nx.Graph()
for _, row in edge_weights.iterrows():
    G_static.add_edge(
        row["source"], row["target"],
        weight=max(row["total_planned"], 1)
    )

# Centrality measures computed once on the training-period graph
degree_c      = dict(G_static.degree())
betweenness_n = nx.betweenness_centrality(G_static, normalized=True, weight="weight")
closeness     = nx.closeness_centrality(G_static, distance="weight")
clustering    = nx.clustering(G_static, weight="weight")
edge_btwn     = nx.edge_betweenness_centrality(G_static, normalized=True, weight="weight")

try:
    eigenvector = nx.eigenvector_centrality(G_static, max_iter=1000, weight="weight")
except nx.PowerIterationFailedConvergence:
    print("Warning: eigenvector centrality did not converge — defaulting to 0.")
    eigenvector = {n: 0.0 for n in G_static.nodes()}


def get_topo_row(src, tgt):
    """Return topology features for one (source, target) pair."""
    edge_key   = (src, tgt) if G_static.has_edge(src, tgt) else (tgt, src)
    common_nbr = (
        len(list(nx.common_neighbors(G_static, src, tgt)))
        if G_static.has_node(src) and G_static.has_node(tgt) else 0
    )
    return {
        "topo_src_degree":        degree_c.get(src, 0),
        "topo_tgt_degree":        degree_c.get(tgt, 0),
        "topo_src_betweenness":   betweenness_n.get(src, 0),
        "topo_tgt_betweenness":   betweenness_n.get(tgt, 0),
        "topo_src_closeness":     closeness.get(src, 0),
        "topo_tgt_closeness":     closeness.get(tgt, 0),
        "topo_src_clustering":    clustering.get(src, 0),
        "topo_tgt_clustering":    clustering.get(tgt, 0),
        "topo_src_eigenvector":   eigenvector.get(src, 0),
        "topo_tgt_eigenvector":   eigenvector.get(tgt, 0),
        "topo_edge_betweenness":  edge_btwn.get(edge_key, 0),
        "topo_common_neighbours": common_nbr,
    }

topo_rows = panel.apply(lambda r: get_topo_row(r["source"], r["target"]), axis=1)
topo_df   = pd.DataFrame(topo_rows.tolist(), index=panel.index)
panel     = pd.concat([panel, topo_df], axis=1)


# %%
# ---------------------------------------------------------------------------
# 12. Integer time_id (needed by Tier 2 & 3)
# ---------------------------------------------------------------------------
period_map       = {p: i for i, p in enumerate(periods_all)}
panel["time_id"] = panel["YearMonth"].map(period_map)


# %%
# ---------------------------------------------------------------------------
# 13. Fill remaining NaNs & save
#
#     NOT done here (done in 01_tier1_model.py after the split):
#       - Binarising proportion_disrupted -> Significant_Disruption
#       - Computing the threshold tau (50th pct of training distribution)
# ---------------------------------------------------------------------------
panel = panel.fillna(0)

panel.to_parquet(OUT_FILE, index=False)
print(f"\nPanel saved -> {OUT_FILE}")
print(f"Shape: {panel.shape}")
print(f"\nColumns:\n{list(panel.columns)}")


Loading data ...
Min-rides filter (>= 4): 51877 -> 39928 rows (11949 dropped, 23.0 %)
Topology graph: built on 50/72 months (up to and including 2023-02)

Panel saved -> C:\Users\EduardCP\Documents\GitHub\MasterThesis\data\processed\modelling_panel_THRESHOLD.parquet
Shape: (39928, 85)

Columns:
['source', 'target', 'YearMonth', 'disrupted_days', 'rides_planned', 'DR', 'RH', 'SQ', 'TG', 'TN', 'TX', 'RHX', 'FHVEC', 'VVX', 'T10N', 'source_WorkHistory_NotEmployed_Last4Y_pct', 'source_WorkHistory_ConstantlyEmployed_Last4Y_pct', 'source_WorkHistory_Retired_pct', 'source_Wealth_Percentile_1_40_pct', 'source_Income_Avg_Percentile_Score', 'source_Edu_Low_pct', 'source_Edu_High_Total_pct', 'source_PrivateHouseholds_Count', 'source_SES_Score_WorkHistory_Avg', 'source_SES_Score_Wealth_Avg', 'source_SES_Score_Education_Avg', 'source_NetWorth_Avg_Percentile_Score', 'source_Population', 'source_TotalPropertyDamageAndViolence_Index', 'source_BicycleTheft', 'source_TotalVandalism', 'source_TotalViolent

In [25]:
df.head(50)

,Date,source,target,Rides planned,Final arrival delay,Final arrival cancelled,Completely cancelled,Intermediate arrival delays,Statistical Causes,Disruption Count,...,target_TotalVandalism,target_TotalViolentAndSexualCrimes,target_TotalPropertyCrimes_Rate,target_Dist_Secondary_Total,target_Dist_GP_Surgery,target_Dist_Supermarket,target_Dist_Highway_Entrance,target_Dist_Train_Station_Total,target_Dist_Major_Transfer_Station,YearMonth
0,2019-01-01,'s-Hertogenbosch,Arnhem Centraal,1,0,0,0,1,No Disruption,0.0,...,NaN,NaN,NaN,1.523148,1.122222,0.904630,1.857407,2.245370,3.610185,2019-01
1,2019-01-01,'s-Hertogenbosch,Den Haag Centraal,33,4,2,0,27,No Disruption,0.0,...,NaN,NaN,NaN,1.018065,0.746452,0.729677,3.072258,3.429677,4.401290,2019-01
2,2019-01-01,'s-Hertogenbosch,Deurne,17,4,1,0,11,No Disruption,0.0,...,NaN,NaN,NaN,4.094595,1.540541,1.400000,2.143243,4.013514,12.345946,2019-01
3,2019-01-01,'s-Hertogenbosch,Dordrecht,11,0,0,0,3,No Disruption,0.0,...,NaN,NaN,NaN,1.708130,1.154472,0.965854,1.583740,2.098374,3.191870,2019-01
4,2019-01-01,'s-Hertogenbosch,Eindhoven Centraal,18,2,0,0,7,No Disruption,0.0,...,NaN,NaN,NaN,1.478030,1.012121,0.875758,2.623485,3.403788,3.912121,2019-01
5,2019-01-01,'s-Hertogenbosch,Roosendaal,2,0,1,1,0,No Disruption,1.0,...,NaN,NaN,NaN,3.189855,1.288406,1.475362,1.805797,3.844928,4.128986,2019-01
6,2019-01-01,'s-Hertogenbosch,Utrecht Centraal,1,0,0,0,1,"['damaged overhead wires', 'an emergency call']",2.0,...,NaN,NaN,NaN,1.418333,0.819167,0.790833,2.185000,2.095833,4.259167,2019-01
7,2019-01-01,Alkmaar,Amsterdam Centraal,6,0,0,0,2,No Disruption,0.0,...,NaN,NaN,NaN,3.555734,1.695079,1.578807,2.008171,5.963448,13.379274,2019-01
8,2019-01-01,Alkmaar,Hoorn,1,0,0,0,0,No Disruption,0.0,...,NaN,NaN,NaN,1.542683,0.819512,0.759756,1.879268,1.981707,3.015854,2019-01
9,2019-01-01,Alkmaar,Maastricht,20,3,0,0,12,No Disruption,0.0,...,NaN,NaN,NaN,2.211765,0.980392,0.923529,2.717647,2.919608,3.433333,2019-01


In [26]:
panel

,source,target,YearMonth,disrupted_days,rides_planned,DR,RH,SQ,TG,TN,...,topo_tgt_betweenness,topo_src_closeness,topo_tgt_closeness,topo_src_clustering,topo_tgt_clustering,topo_src_eigenvector,topo_tgt_eigenvector,topo_edge_betweenness,topo_common_neighbours,time_id
0,'s-Hertogenbosch,Alkmaar,2019-05,0,122,11.285714,15.285714,51.000000,99.857143,49.000000,...,0.020911,0.013939,0.012679,0.003140,0.002713,0.096350,0.038308,0.0,20,4
1,'s-Hertogenbosch,Alkmaar,2019-10,1,43,39.500000,32.500000,4.000000,112.500000,87.500000,...,0.020911,0.013939,0.012679,0.003140,0.002713,0.096350,0.038308,0.0,20,9
2,'s-Hertogenbosch,Alkmaar,2019-11,0,20,0.000000,0.000000,71.000000,16.000000,-26.000000,...,0.020911,0.013939,0.012679,0.003140,0.002713,0.096350,0.038308,0.0,20,10
3,'s-Hertogenbosch,Alkmaar,2019-12,0,22,0.000000,0.000000,7.000000,11.000000,-22.000000,...,0.020911,0.013939,0.012679,0.003140,0.002713,0.096350,0.038308,0.0,20,11
4,'s-Hertogenbosch,Alkmaar,2020-01,0,22,0.000000,0.000000,52.000000,59.000000,12.000000,...,0.020911,0.013939,0.012679,0.003140,0.002713,0.096350,0.038308,0.0,20,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39923,Zürich HB,Frankfurt (M) Hbf,2024-02,0,29,33.034483,33.413793,22.034483,71.517241,51.827586,...,0.011939,0.001963,0.013775,0.012986,0.002183,0.001034,0.017906,0.0,1,61
39924,Zürich HB,Frankfurt (M) Hbf,2024-03,3,31,14.677419,10.677419,40.032258,77.677419,50.774194,...,0.011939,0.001963,0.013775,0.012986,0.002183,0.001034,0.017906,0.0,1,62
39925,Zürich HB,Frankfurt (M) Hbf,2024-04,0,30,19.500000,26.200000,56.900000,97.966667,67.400000,...,0.011939,0.001963,0.013775,0.012986,0.002183,0.001034,0.017906,0.0,1,63
39926,Zürich HB,Frankfurt (M) Hbf,2024-05,0,31,13.161290,21.290323,84.387097,147.258065,109.032258,...,0.011939,0.001963,0.013775,0.012986,0.002183,0.001034,0.017906,0.0,1,64
